# 01 — EDA
Look at the data: shape, class balance, and what the spectrograms look like for 6 vs 7.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # so `import src...` works from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import load_train, load_test, to_grayscale, to_images

In [ ]:
ids, X_raw, y = load_train()
X = to_grayscale(X_raw)
print(X_raw.shape, '->', X.shape)
print(pd.Series(y).value_counts())

In [ ]:
imgs = to_images(X)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for row, label in enumerate([0, 1]):
    for ax, i in zip(axes[row], np.where(y == label)[0][:5]):
        ax.imshow(imgs[i], cmap='gray', origin='lower')
        ax.set_title(f'ID {ids[i]} ({"6" if label == 0 else "7"})')
        ax.axis('off')
plt.tight_layout()

In [ ]:
# Mean spectrogram per class and their difference
mean6, mean7 = imgs[y == 0].mean(0), imgs[y == 1].mean(0)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(axes, [mean6, mean7, mean7 - mean6], ['mean 6', 'mean 7', '7 - 6']):
    ax.imshow(img, cmap='gray' if title != '7 - 6' else 'RdBu', origin='lower'); ax.set_title(title); ax.axis('off')
plt.savefig('../results/figures/class_means.png', dpi=150, bbox_inches='tight')

In [ ]:
# How many PCA components explain most of the variance?
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
p = PCA().fit(StandardScaler().fit_transform(X))
plt.plot(np.cumsum(p.explained_variance_ratio_)); plt.xlabel('components'); plt.ylabel('cumulative variance'); plt.grid()